<a href="https://colab.research.google.com/github/nazndev/visibility-gated-clip-reid/blob/main/notebooks/colab_iteration2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import subprocess
from pathlib import Path

REVISION = ""  # blank locks origin/main; or paste a 40-character commit
REPO = "https://github.com/nazndev/visibility-gated-clip-reid.git"
ROOT = Path("/content/lvlm-person-reid")

if not (ROOT / "src" / "reidvlm").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
subprocess.check_call(["git", "fetch", "origin"], cwd=ROOT)
if not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    REVISION = subprocess.check_output(["git", "rev-parse", "origin/main"], cwd=ROOT, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    raise SystemExit("origin/main is not a 40-character commit.")
subprocess.check_call(["git", "checkout", "--force", REVISION], cwd=ROOT)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != REVISION:
    raise SystemExit(f"HEAD {head} is not {REVISION}")
os.chdir(ROOT)
print(head)


5d1c0c0faf6cd5246ad0d3c515e98076f755bb5e


In [2]:
import os

print("nvidia device", os.path.exists("/dev/nvidia0"))
if not os.path.exists("/dev/nvidia0"):
    raise RuntimeError("Select a GPU runtime, then run this cell again.")


nvidia device False


RuntimeError: Select a GPU runtime, then run this cell again.

In [ ]:
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
subprocess.check_call(["pip", "install", "-q", "-r", "requirements-colab.txt"], cwd=ROOT)
subprocess.check_call(["pip", "install", "-q", "-e", "."], cwd=ROOT)
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda"
print(torch.__version__, torch.cuda.is_available(), name)


In [ ]:
import subprocess
from pathlib import Path
from google.colab import drive

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2")
drive.mount("/content/drive")
DRIVE.mkdir(parents=True, exist_ok=True)
if not (ROOT / "data/Market-1501-v15.09.15/bounding_box_train").is_dir():
    subprocess.check_call(["python", "scripts/01_download_dataset.py"], cwd=ROOT)
if not (ROOT / "checkpoints/Market1501_clipreid_ViT-B-16_60.pth").is_file():
    subprocess.check_call(["python", "scripts/02_download_model.py"], cwd=ROOT)
print(DRIVE)


In [ ]:
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2")
subprocess.check_call(["python", "-m", "reidvlm", "prune-plan", "--output", str(DRIVE / "iteration2_plan.json")], cwd=ROOT)
subprocess.check_call(["python", "-m", "reidvlm", "prune-manifest", "--config", "configs/market1501.yaml", "--output", str(DRIVE / "validation_manifest.json")], cwd=ROOT)
print("plan and manifest written")


In [ ]:
# Change these, then run this cell once. Do not start a second copy while it is running.
STRATEGY = "random"  # none, cls_attention, random
BLOCK = "6"
KEEP = "64"
SEED = "11"  # required for random: 11, 23, 37, 53, or 71
SPLIT = "validation"  # validation uses training images; test is the official split
BACKEND = "reference"  # reference or cls_row

import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2")
if STRATEGY == "random" and SEED not in {"11", "23", "37", "53", "71"}:
    raise SystemExit("Random selection needs SEED set to 11, 23, 37, 53, or 71.")
name = f"{SPLIT}_{STRATEGY}_b{BLOCK}_k{KEEP}_{BACKEND}" + (f"_s{SEED}" if STRATEGY == "random" else "")
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", SPLIT,
    "--strategy", STRATEGY,
    "--prune-block", BLOCK,
    "--keep", KEEP,
    "--backend", BACKEND,
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
if STRATEGY == "random":
    command.extend(["--seed", SEED])
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)


In [ ]:
import json
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2")
rows = []
for path in sorted(DRIVE.glob("*.json")):
    if path.name in {"iteration2_plan.json", "validation_manifest.json"}:
        continue
    payload = json.loads(path.read_text())
    scores = payload.get("scores", {})
    rows.append({
        "file": path.name,
        "label": payload.get("label"),
        "mAP": scores.get("mAP"),
        "rank1": scores.get("rank1"),
        "kreciprocal": payload.get("kreciprocal"),
    })
print(json.dumps(rows, indent=2))
